<a href="https://colab.research.google.com/github/amitphadke1405/fastText-TextClassification-Implementation/blob/main/fastText_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q datasets

In [6]:
import re, zlib, time, random
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from datasets import load_dataset

hidden_size = 10
num_epochs = 5
batch_size = 32
use_bigrams = True
num_buckets = 2000000
lr_options = [0.5, 1.0, 2.0, 5.0]

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

Running on: cpu


In [7]:
data = load_dataset("fancyzhx/ag_news")

all_texts = data["train"]["text"]
all_labels = data["train"]["label"]
test_texts = data["test"]["text"]
test_labels = data["test"]["label"]
num_classes = len(set(all_labels))

#10% of the training data aside for validation
shuffled = np.random.permutation(len(all_texts))
val_size = len(all_texts) // 10
val_ids = shuffled[:val_size]
train_ids = shuffled[val_size:]

train_texts = [all_texts[i] for i in train_ids]
train_labels = [all_labels[i] for i in train_ids]
val_texts = [all_texts[i] for i in val_ids]
val_labels = [all_labels[i] for i in val_ids]

print("train:", len(train_texts), "val:", len(val_texts), "test:", len(test_texts))

train: 108000 val: 12000 test: 7600


In [9]:
def split_words(text):
    return re.findall(r"[a-z0-9']+", text.lower())

# vocabulary from the training texts only
word_counts = Counter()
for text in train_texts:
    word_counts.update(split_words(text))

word_to_id = {word: i for i, word in enumerate(word_counts)}
vocab_size = len(word_to_id)
total_rows = vocab_size + (num_buckets if use_bigrams else 0)
print("vocab size:", vocab_size, "| total embedding rows:", total_rows)

def text_to_ids(text):
    words = split_words(text)
    ids = [word_to_id[w] for w in words if w in word_to_id]
    if use_bigrams:
        for first, second in zip(words, words[1:]):
            bucket = zlib.crc32((first + " " + second).encode()) % num_buckets
            ids.append(vocab_size + bucket)
    return ids

train_data = [text_to_ids(t) for t in train_texts]
val_data = [text_to_ids(t) for t in val_texts]
test_data = [text_to_ids(t) for t in test_texts]
print("converting doneeee....Yaay")

vocab size: 67638 | total embedding rows: 2067638
converting doneeee....Yaay


In [10]:
def get_batch(data_list, label_list, indices):
    chosen = [data_list[i] for i in indices]
    lengths = [len(ids) for ids in chosen]
    starts = torch.tensor([0] + np.cumsum(lengths)[:-1].tolist())
    flat_ids = torch.tensor([x for ids in chosen for x in ids])
    targets = torch.tensor([label_list[i] for i in indices])
    return flat_ids.to(device), starts.to(device), targets.to(device)


class FastText(nn.Module):
    def __init__(self, num_rows, hidden_size, num_classes):
        super().__init__()
        self.word_vectors = nn.EmbeddingBag(num_rows, hidden_size, mode="mean", sparse=True)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, flat_ids, starts):
        return self.classifier(self.word_vectors(flat_ids, starts))


def get_accuracy(model, data_list, label_list):
    correct = 0
    with torch.no_grad():
        for start in range(0, len(data_list), 1024):
            indices = range(start, min(start + 1024, len(data_list)))
            flat_ids, starts, targets = get_batch(data_list, label_list, indices)
            correct += (model(flat_ids, starts).argmax(dim=1) == targets).sum().item()
    return correct / len(data_list)

In [11]:
learning_rate = 2.0

model = FastText(total_rows, hidden_size, num_classes).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

steps_per_epoch = len(train_data) // batch_size
scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1.0, end_factor=0.0, total_iters=num_epochs * steps_per_epoch
)

for epoch in range(num_epochs):
    order = np.random.permutation(len(train_data))
    total_loss = 0

    for step in range(steps_per_epoch):
        batch_ids = order[step * batch_size : (step + 1) * batch_size]
        flat_ids, starts, targets = get_batch(train_data, train_labels, batch_ids)

        loss = loss_function(model(flat_ids, starts), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    val_acc = get_accuracy(model, val_data, val_labels)
    print(f"epoch {epoch + 1}: loss {total_loss / steps_per_epoch:.4f}, validation accuracy {val_acc:.4f}")

test_acc = get_accuracy(model, test_data, test_labels)
print(f"\nTest accuracy (bigrams={use_bigrams}): {test_acc * 100:.2f}%")

epoch 1: loss 0.9344, validation accuracy 0.7492
epoch 2: loss 0.5529, validation accuracy 0.8132
epoch 3: loss 0.4673, validation accuracy 0.8325
epoch 4: loss 0.4304, validation accuracy 0.8411
epoch 5: loss 0.4137, validation accuracy 0.8432

Test accuracy (bigrams=True): 84.22%
